In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

OPENAI_API_KEY = os.environ['OPENAI_API_KEY']
#PINECONE_API_KEY = os.environ['PINECONE_API_KEY']

print("API 키 로드 완료")
print(f"OpenAI Key: {OPENAI_API_KEY[:10]}...")

API 키 로드 완료
OpenAI Key: 여기에_본인키_입력...


In [4]:
# Step 2 : LCEL 기본 체인 동작 확인
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini")
prompt = PromptTemplate.from_template("{topic}에 대해 한 문장으로 설명해줘")
chain = prompt | llm | StrOutputParser()

result = chain.invoke({"topic": "홍민택"})
print(result)

홍민택은 한국의 기업가이자 인공지능 기술 분야에서 활동하는 전문가로, 혁신적인 솔루션 개발에 기여하고 있습니다.


In [3]:
import langchain
print(langchain.__version__)

1.2.15


In [5]:
# 01_simple_agent.py

from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent   # LangChain 1.x
from langgraph.prebuilt import create_react_agent

# 1. 도구 정의
@tool
def add(a: float, b: float) -> float:
    """두 숫자를 더합니다."""
    return a + b

@tool
def multiply(a: float, b: float) -> float:
    """두 숫자를 곱합니다."""
    return a * b

tools = [add, multiply]

# 2. 에이전트 생성
agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt="당신은 계산을 도와주는 어시스턴트입니다."  
)
# 3. 실행
result = agent.invoke({
    "messages": [{"role": "user", "content": "3 더하기 5는 얼마야? 그리고 그 결과에 7을 곱하면?"}]
})

print(result["messages"][-1].content)

C:\Users\Admin\AppData\Local\Temp\ipykernel_18632\619990027.py:25: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


3 더하기 5는 8입니다. 그리고 그 결과에 7을 곱하면 56이 됩니다.


In [6]:
# 도구 호출 과정을 단계별로 확인
for msg in result["messages"]:
    print(f"[{msg.__class__.__name__}] {msg.content[:80] if msg.content else '(tool call)'}")

[HumanMessage] 3 더하기 5는 얼마야? 그리고 그 결과에 7을 곱하면?
[AIMessage] (tool call)
[ToolMessage] 8.0
[ToolMessage] 56.0
[AIMessage] 3 더하기 5는 8입니다. 그리고 그 결과에 7을 곱하면 56이 됩니다.


In [10]:
! pip install -U langchain-tavily


In [9]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
# 2026년 표준: 통합 create_agent만 가져옵니다.
from langchain.agents import create_agent 

import ast
import operator

# 1. 도구 정의 (검색 및 계산기 - 이전과 동일)
search_tool = TavilySearchResults(max_results=3)

SAFE_OPERATORS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv, ast.Pow: operator.pow}
def _safe_eval(node):
    if isinstance(node, ast.Constant): return node.value
    elif isinstance(node, ast.BinOp):
        op = SAFE_OPERATORS.get(type(node.op))
        return op(_safe_eval(node.left), _safe_eval(node.right)) if op else None
    raise ValueError("허용되지 않는 표현식")

@tool
def calculate(expression: str) -> str:
    """사칙연산과 거듭제곱 전용 안전 계산기. 예: '1234 * 5678'"""
    try:
        tree = ast.parse(expression, mode="eval")
        return str(_safe_eval(tree.body))
    except Exception as e: return f"계산 오류: {e}"

# 2. 에이전트 생성 (2026년 최신 방식)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 'create_tool_calling_agent' 대신 'create_agent'를 사용합니다.
# 'prompt' 대신 'system_prompt'를 사용하며, scratchpad 등은 자동으로 처리됩니다.
agent = create_agent(
    model=llm,
    tools=[search_tool, calculate],
    system_prompt="당신은 웹 검색과 계산을 도와주는 유능한 AI 어시스턴트입니다."
)

# 3. 테스트 실행
# 최신 에이전트는 'input' 대신 'messages' 리스트 형식을 기본으로 사용합니다.
print("\n=== 테스트: 검색 + 계산기 통합 사용 ===")
response = agent.invoke({
    "messages": [
        ("user", "3+5 를 하고 *7를 하면 ?")
    ]
})

# 결과 출력 (최신 버전은 마지막 메시지를 확인합니다)
print(f"답변: {response['messages'][-1].content}")


=== 테스트: 검색 + 계산기 통합 사용 ===
답변: \( (3 + 5) \times 7 = 56 \)입니다.


In [12]:
result3 = agent.invoke({
     "messages": [{"role": "user", "content": "2024년 한국 GDP를 검색하고, 그 금액을 1300으로 나누면 얼마야?"}]
 })

for msg in result3["messages"]:
    name = msg.__class__.__name__
    # 툴 호출 정보 추출
    tool_calls = getattr(msg, "tool_calls", [])
    if tool_calls:
        for tc in tool_calls:
            print(f"[{name}] 도구 호출: {tc['name']}({tc['args']})")
    else:
        content = str(msg.content)[:120] if msg.content else ""
        print(f"[{name}] {content}")

[HumanMessage] 2024년 한국 GDP를 검색하고, 그 금액을 1300으로 나누면 얼마야?
[AIMessage] 도구 호출: tavily_search_results_json({'query': '2024년 한국 GDP'})
[ToolMessage] [{"title": "2024년 한국 GDP, 일본ㆍ대만 제쳤다 - 이코노믹리뷰", "url": "https://www.econovill.com/news/articleView.html?idxno=682047", "c
[AIMessage] 도구 호출: calculate({'expression': '36024 / 1300'})
[ToolMessage] 27.71076923076923
[AIMessage] 2024년 한국의 1인당 GDP는 약 36,024달러로 추산됩니다. 이 금액을 1,300으로 나누면 약 27.71이 됩니다.


In [13]:
for step in agent.stream(
    {"messages": [{"role": "user", "content": "2024년 한국 GDP를 검색하고, 그 금액을 1300으로 나누면 얼마야?"}]},
    stream_mode="updates",   # 각 노드 업데이트마다 출력
):
    for node_name, node_output in step.items():
        print(f"\n{'='*40}")
        print(f"노드: {node_name}")
        for msg in node_output.get("messages", []):
            name = msg.__class__.__name__
            tool_calls = getattr(msg, "tool_calls", [])
            if tool_calls:
                for tc in tool_calls:
                    print(f"  도구 호출: {tc['name']}({tc['args']})")
            else:
                print(f"  [{name}] {str(msg.content)[:120]}")


노드: model
  도구 호출: tavily_search_results_json({'query': '2024 South Korea GDP'})

노드: tools
  [ToolMessage] [{"title": "S.Korea's economy grows 2.0% in 2024; Q4 GDP below forecast", "url": "https://www.kedglobal.com/economy/news

노드: model
  도구 호출: calculate({'expression': '1917.30 / 1300'})

노드: tools
  [ToolMessage] 1.474846153846154

노드: model
  [AIMessage] 2024년 한국의 GDP는 약 1917.30억 달러입니다. 이 금액을 1300으로 나누면 약 1.47입니다.


In [14]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_community.tools.tavily_search import TavilySearchResults

load_dotenv()

# 1. 부장님 모드 포함 전문가 프롬프트 정의
PROMPT_TEMPLATES = {
    # [핵심] 전략기획실 부장님 모드
    "manager_report": """당신은 대기업 전략기획실의 20년 경력 '이부장'입니다. 
모든 답변은 상급자에게 보고하기 위한 '완결된 보고서' 형식으로 작성하세요. 
다음 지침을 반드시 준수하십시오:
1. 반드시 '개조식(1., -, *)'을 사용하여 가독성을 높일 것.
2. '추진 배경 - 현황 및 문제점 - 검토 의견 - 시사점'의 구조를 갖출 것.
3. 문장 끝은 '~함', '~임', '~ 요망' 등 보고서체(명사형 종결)를 사용할 것.
4. 전문 용어를 적절히 섞어 신뢰감을 주되 핵심 위주로 요약할 것.""",

    "creative_writer": """당신은 감성적이고 창의적인 카피라이터입니다. 
친근하고 부드러운 말투로 독자의 마음을 사로잡는 글을 쓰세요. 
비유와 형용사를 풍부하게 사용하십시오.""",

    "general": "당신은 유능한 비서입니다. 사용자의 질문에 짧고 명확하게 답하세요."
}

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
search_tool = TavilySearchResults(max_results=3)

# 2. 라우터 함수 (의도 분류)
def route_query(query: str) -> str:
    classification_prompt = f"""다음 질문이 '보고서 작성'이나 '업무 분석' 관련이면 'manager_report', 
'창의적 글쓰기'나 '감성 문구' 관련이면 'creative_writer', 그 외엔 'general'로 분류하세요.
단어 하나만 대답하세요.

질문: {query}"""
    
    category = llm.invoke(classification_prompt).content.strip().lower()
    return category if category in PROMPT_TEMPLATES else "general"

# 3. 에이전트 실행 함수
def run_office_agent(user_input: str):
    category = route_query(user_input)
    print(f"📡 [시스템] {category} 모드로 전환하여 보고를 시작합니다.\n")
    
    selected_prompt = PROMPT_TEMPLATES[category]
    
    # 2026년형 통합 에이전트 생성
    agent = create_agent(
        model=llm,
        tools=[search_tool],
        system_prompt=selected_prompt
    )
    
    response = agent.invoke({
        "messages": [("user", user_input)]
    })
    return response["messages"][-1].content

# 4. 테스트 실행 (부장님 모드 호출)
report_query = "최근 AI 트렌드와 우리 회사가 도입해야 할 이유에 대해 정리해봐."
final_report = run_office_agent(report_query)

print("-" * 50)
print(final_report)
print("-" * 50)

📡 [시스템] manager_report 모드로 전환하여 보고를 시작합니다.

--------------------------------------------------
**보고서: 2023년 AI 트렌드 및 도입 필요성**

1. **추진 배경**
   - 최근 AI 기술의 발전이 가속화되고 있으며, 특히 생성적 AI(Generative AI)와 적응형 AI(Adaptive AI)가 주목받고 있음.
   - OpenAI의 ChatGPT와 같은 혁신적인 AI 모델들이 시장에 출시되면서, 대기업들이 AI 도입에 적극적으로 나서고 있음.
   - AI는 업무 효율성을 높이고, 고객 경험을 개선하는 데 중요한 역할을 할 것으로 예상됨.

2. **현황 및 문제점**
   - 현재 AI 기술은 다양한 산업에서 활용되고 있으나, 우리 회사는 AI 도입이 미비하여 경쟁사 대비 뒤처질 위험이 있음.
   - AI 도입을 위한 인프라와 인력 교육이 부족하여, 기술 활용에 대한 저항감이 존재함.
   - AI의 발전으로 인해 일부 직무가 사라질 가능성이 있으나, 새로운 직무가 창출될 것이라는 점을 간과해서는 안 됨.

3. **검토 의견**
   - AI 도입을 통해 업무 프로세스를 자동화하고, 데이터 분석을 통한 의사결정 지원이 가능함.
   - 고객 맞춤형 서비스 제공을 위한 AI 기반 추천 시스템 도입이 필요함.
   - AI 기술을 활용한 보안 시스템 강화로 사이버 공격에 대한 실시간 대응이 가능해짐.
   - 직원 교육 및 AI 활용 방안에 대한 체계적인 프로그램 마련이 요구됨.

4. **시사점**
   - AI 도입은 단순한 기술적 변화가 아니라, 기업의 경쟁력을 좌우하는 중요한 요소임.
   - 시장의 변화에 발맞추어 AI 기술을 적극적으로 도입하고 활용해야 함.
   - 향후 AI 기술의 발전에 따라 지속적인 투자와 연구개발이 필요함.

**결론**: AI 도입은 우리 회사의 미래 성장과 경쟁력 강화를 위한 필수적인 전략임. 이에 대한 신속한 검토와 실행을 요